In [1]:
from src.dataset_manager import DatasetManager
from src.models.cox import CoxPiecewise
from src.training_manager import GGSTrainingManager
from src.test_manager import TestManager

/home/jdani/proyects/Premant/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
m_train, m_test = DatasetManager.split_dataset()

Training engines: 140
Test engines: 60


In [3]:
Training = GGSTrainingManager(
    model=CoxPiecewise(),
    list_ids=m_train
)

Test = TestManager(
    model=CoxPiecewise(),
    train_ids=m_train,
    test_ids=m_test
)

In [11]:
param_grid = {
    'alphas': [0.001, 0.005, 0.01, 0.05, 0.08, 0.12],
    'l1_ratio': [0.5, 0.95, 1.0],
    'confidence_threshold': [0.5, 0.7, 0.9],
    'clipping_threshold': [110, 115, 120, 125]
}

ggs = Training.group_grid_search(param_grid=param_grid)
display(Training.get_ggs_results())

Starting Grid Search: 216 configs × 5 folds = 1080 tasks


GGS progress: 100%|██████████| 216/216 [27:54<00:00,  7.75s/it]


,alphas,l1_ratio,confidence_threshold,clipping_threshold,mean_S_score,mean_C_index,mean_MAE,mean_RMSE,Success
0,0.001,0.50,0.5,110,1947.686673,0.5,22.483457,39.667066,1
1,0.001,0.50,0.7,110,1947.686673,0.5,22.483457,39.667066,1
2,0.001,0.50,0.9,110,1947.686673,0.5,22.483457,39.667066,1
3,0.001,0.95,0.5,110,1947.686673,0.5,22.483457,39.667066,1
4,0.001,0.95,0.7,110,1947.686673,0.5,22.483457,39.667066,1
5,0.001,0.95,0.9,110,1947.686673,0.5,22.483457,39.667066,1
6,0.001,1.00,0.5,110,1947.686673,0.5,22.483457,39.667066,1
7,0.001,1.00,0.7,110,1947.686673,0.5,22.483457,39.667066,1
8,0.001,1.00,0.9,110,1947.686673,0.5,22.483457,39.667066,1
9,0.005,0.50,0.5,110,1947.686673,0.5,22.483457,39.667066,1


In [9]:
display(Training.get_ggs_results(50))

,alphas,l1_ratio,confidence_threshold,clipping_threshold,mean_S_score,mean_C_index,mean_MAE,mean_RMSE,Success
0,0.001,0.50,0.5,110,1947.686673,0.5,22.483457,39.667066,1
1,0.001,0.50,0.7,110,1947.686673,0.5,22.483457,39.667066,1
2,0.001,0.50,0.9,110,1947.686673,0.5,22.483457,39.667066,1
3,0.001,0.95,0.5,110,1947.686673,0.5,22.483457,39.667066,1
4,0.001,0.95,0.7,110,1947.686673,0.5,22.483457,39.667066,1
5,0.001,0.95,0.9,110,1947.686673,0.5,22.483457,39.667066,1
6,0.001,1.00,0.5,110,1947.686673,0.5,22.483457,39.667066,1
7,0.001,1.00,0.7,110,1947.686673,0.5,22.483457,39.667066,1
8,0.001,1.00,0.9,110,1947.686673,0.5,22.483457,39.667066,1
9,0.005,0.50,0.5,110,1947.686673,0.5,22.483457,39.667066,1


In [4]:
from src.models.cox import CoxPiecewise
import numpy as np

X, y_fit, y_metrics, groups = CoxPiecewise().prepare_training_data(m_train[:10])

model = CoxPiecewise(alphas=0.001, l1_ratio=0.5, confidence_threshold=0.5, clipping_threshold=110)
from sklearn.preprocessing import RobustScaler
X_scaled = RobustScaler().fit_transform(X)
model.fit(X_scaled, y_fit)

print("is_fitted:", model.is_fitted_)
print("model_.alphas_:", model.model_.alphas_)
print("self.alphas:", model.alphas)
print("alpha_value usado en predict:", float(model.alphas))

is_fitted: True
model_.alphas_: [0.001]
self.alphas: 0.001
alpha_value usado en predict: 0.001


In [5]:
from src.training_manager import _run_single_config
from src.models.cox import CoxPiecewise

params = {
    'alphas': 0.001,
    'l1_ratio': 0.5,
    'confidence_threshold': 0.5,
    'clipping_threshold': 110
}

result = _run_single_config(
    params=params,
    model_class=CoxPiecewise,
    X=Training.X_train,
    y_fit=Training.y_fit_train,
    y_metrics=Training.y_metrics_train,
    groups=Training.groups_train,
    n_folds=2
)

print(result)

{'alphas': 0.001, 'l1_ratio': 0.5, 'confidence_threshold': 0.5, 'clipping_threshold': 110, 'mean_S_score': 1947.6911661463441, 'mean_C_index': 0.5, 'mean_MAE': 22.483370357347507, 'mean_RMSE': 39.66505741381059}


In [6]:
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from src.models.cox import CoxPiecewise
from src.metrics_manager import Metrics

params = {'alphas': 0.001, 'l1_ratio': 0.5, 'confidence_threshold': 0.5, 'clipping_threshold': 110}

gkf = GroupKFold(n_splits=2)
X = Training.X_train
y_fit = Training.y_fit_train
y_metrics = Training.y_metrics_train
groups = Training.groups_train

for train_idx, val_idx in gkf.split(X, y_fit, groups):
    X_train_fold = X.iloc[train_idx]
    X_val_fold = X.iloc[val_idx]
    y_fit_train = y_fit[train_idx]
    y_metrics_val = y_metrics[val_idx]

    model = CoxPiecewise(**params)
    pipeline = Pipeline([('scaler', RobustScaler()), ('model', model)])
    pipeline.fit(X_train_fold, y_fit_train)
    
    print("is_fitted:", model.is_fitted_)
    print("model_.alphas_:", model.model_.alphas_)
    
    # Ver predicciones
    preds = pipeline.predict(X_val_fold)
    print("preds sample:", preds[:5])
    print("preds unique:", len(set(preds.round(2))))
    
    # Ver métricas
    metrics_funcs = Metrics.get_metrics()
    for name, func in metrics_funcs.items():
        try:
            result = func(pipeline, X_val_fold, y_metrics_val)
            print(f"{name}: {result}")
        except Exception as e:
            print(f"{name} FAILED: {type(e).__name__}: {e}")
    break

is_fitted: True
model_.alphas_: [0.001]
preds sample: [110. 110. 110. 110. 110.]
preds unique: 1
S_score: 2013.7107870401494
C_index: 0.5
MAE: 23.391067080849634
RMSE: 40.44883873632367


In [7]:
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from src.models.cox import CoxPiecewise
import numpy as np

params = {'alphas': 0.001, 'l1_ratio': 0.5, 'confidence_threshold': 0.5, 'clipping_threshold': 110}

gkf = GroupKFold(n_splits=2)
X = Training.X_train
y_fit = Training.y_fit_train
groups = Training.groups_train

for train_idx, val_idx in gkf.split(X, y_fit, groups):
    X_train_fold = X.iloc[train_idx]
    X_val_fold = X.iloc[val_idx]
    y_fit_train = y_fit[train_idx]

    model = CoxPiecewise(**params)
    pipeline = Pipeline([('scaler', RobustScaler()), ('model', model)])
    pipeline.fit(X_train_fold, y_fit_train)

    # Ver survival functions directamente
    X_val_scaled = pipeline.named_steps['scaler'].transform(X_val_fold)
    sfs = model.model_.predict_survival_function(X_val_scaled, alpha=0.001)
    
    print(f"Num survival functions: {len(sfs)}")
    for i, sf in enumerate(sfs[:3]):
        times = sf.x
        probs = sf(times)
        failure = 1 - probs
        print(f"\nRow {i}:")
        print(f"  times: {times}")
        print(f"  survival: {probs}")
        print(f"  failure: {failure}")
        print(f"  max failure: {failure.max():.4f}")
    break

Num survival functions: 11911

Row 0:
  times: [ 34.  37.  49.  55.  68.  74.  78.  97. 101. 106. 121. 133. 134. 135.
 145. 146. 147. 150. 152. 153. 155. 156. 158. 160. 162. 165. 168. 170.
 172. 174. 177. 180. 181. 186. 188. 189. 192. 194. 195. 196. 198. 200.
 201. 203. 207. 208. 213. 214. 216. 217. 231. 234. 240. 257. 267. 278.
 283. 287.]
  survival: [1.         1.         1.         1.         1.         1.
 1.         1.         1.         1.         1.         1.
 1.         1.         1.         1.         0.98459342 0.98459342
 0.98459342 0.96780717 0.96780717 0.95042598 0.93284923 0.93284923
 0.93284923 0.91312263 0.89272242 0.87207429 0.87207429 0.85045299
 0.85045299 0.82756959 0.80456618 0.80456618 0.78011122 0.75551841
 0.73059551 0.70470059 0.67868307 0.67868307 0.65050849 0.62081311
 0.59100706 0.59100706 0.55911678 0.52719695 0.46400659 0.42916855
 0.39432474 0.35964464 0.29023315 0.29023315 0.24752574 0.20229524
 0.1567175  0.10923708 0.06209868 0.0199379 ]
  failure: [